In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import json

In [2]:
def to_tuple(data):
    if isinstance(data, list):
        return tuple(to_tuple(i) for i in data)
    elif isinstance(data, dict):
        return {k: to_tuple(v) for k, v in data.items()}
    else:
        return data


def read_csv_folder_dict(folder_path):
    """Return a dictionary with filenames as keys."""
    csv_files = list(Path(folder_path).glob("*.csv"))
    csv_files.sort()
    return {file.stem: pd.read_csv(file) for file in csv_files}


def read_json_folder(folder_path):
    """Read all JSON files from a folder into a dict of objects."""
    json_files = list(Path(folder_path).glob("*.json"))
    json_files.sort()
    json_dict = {file.stem: json.load(open(file)) for file in json_files}
    json_dict = to_tuple(json_dict)
    return json_dict


results_dfs = read_csv_folder_dict(
    "./../../data/experiment_output/fair_irl/exp_results/"
)
info_jsons = read_json_folder("./../../data/experiment_output/fair_irl/exp_info/")

organized_data = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(pd.DataFrame)))
)
organized_info = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
)
for filename in results_dfs:
    weight_adjusts = info_jsons[filename]["WEIGHT_ADJUSTS"]
    dataset = info_jsons[filename]["DATASET"]
    expert = info_jsons[filename]["EXPERT_ALGO"]
    bias_types = info_jsons[filename]["BIAS_TYPES"]
    organized_data[dataset][expert][bias_types][weight_adjusts] = results_dfs[filename]
    organized_info[dataset][expert][bias_types][weight_adjusts].append(
        info_jsons[filename]
    )
    pass

# Average across all trials for each dataset, expert, bias type, and weight adjustment
averaged_data = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(pd.DataFrame)))
)
averaged_info = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
)
for dataset in organized_info:
    for expert in organized_info[dataset]:
        for bias_types in organized_info[dataset][expert]:
            for weight_adjusts in organized_info[dataset][expert][bias_types]:
                trial_data = organized_data[dataset][expert][bias_types][weight_adjusts]
                average_series = trial_data.mean()
                averaged_data[dataset][expert][bias_types][
                    weight_adjusts
                ] = average_series
                averaged_info[dataset][expert][bias_types][weight_adjusts] = (
                    organized_info[dataset][expert][bias_types][weight_adjusts][0]
                )
del organized_info
del organized_data
del weight_adjusts
del average_series
del bias_types
del dataset
del expert
del filename
del info_jsons
del results_dfs
del trial_data

In [3]:
selected_expert = "OptAcc"
performance_metric_weight = 0.5
fairness_metric_weight = 1 - performance_metric_weight

unbiased_types = [
    (),
]
biased_types = [
    ("threshold_swapping",),
    ("unbalanced_redlining",),
    ("balanced_redlining",),
    ("perfectly_balanced_redlining",),
]

unadjusted_weights = ()
# adjusted_weights = ()
# adjusted_weights = (("mul_negative_weights", 0.5),)
# adjusted_weights = ("sqrt_negative_weights",)
# adjusted_weights = ("ln_negative_weights",)
adjusted_weights = (
    (("mul_negative_weights", 0.0),),
    (("mul_negative_weights", 0.1),),
    (("mul_negative_weights", 0.2),),
    (("mul_negative_weights", 0.3),),
    (("mul_negative_weights", 0.4),),
    (("mul_negative_weights", 0.5),),
    (("mul_negative_weights", 0.6),),
    (("mul_negative_weights", 0.7),),
    (("mul_negative_weights", 0.8),),
    (("mul_negative_weights", 0.9),),
)

hyperparameter_opt_datasets = [
    "COMPAS",
    "Boston",
    "Adult",
    "ACSIncome__MA",
    "ACSIncome__MS",
]

# The goal is to graph the average fairness expectation error of each weight adjustment on our hyperparameter sweep
# identify what hyperparameter is best

# average across all bias types
# Average across all selected optimization datasets

# List of subdominance metrics
subdom_metric_names = ["max_abs_subdominance", "sum_abs_subdominance", "max_rel_subdominance", "sum_rel_subdominance"]

max_abs_subdom_results_list = []
sum_abs_subdom_results_list = []
max_rel_subdom_results_list = []
sum_rel_subdom_results_list = []

weight_adjustment_strings = []
for weight_adjustment in adjusted_weights:
    weight_adjustment_strings.append(str(weight_adjustment))
    max_abs_subdom_list = []
    sum_abs_subdom_list = []
    max_rel_subdom_list = []
    sum_rel_subdom_list = []
    for dataset in hyperparameter_opt_datasets:
        for biased_type in biased_types:
            max_abs_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["max_abs_subdominance"]
                .item() # TODO: is .item() correct, or should it be .to_numpy() 
            )
            sum_abs_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["sum_abs_subdominance"]
                .item()
            )
            max_rel_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["max_rel_subdominance"]
                .item()
            )
            sum_rel_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["sum_rel_subdominance"]
                .item()
            )                

    max_abs_subdom_results_list.append(np.average(max_abs_subdom_list))
    sum_abs_subdom_results_list.append(np.average(sum_abs_subdom_list))
    max_rel_subdom_results_list.append(np.average(max_rel_subdom_list))
    sum_rel_subdom_results_list.append(np.average(sum_rel_subdom_list))

subdom_metrics_list = [
    max_abs_subdom_results_list,
    sum_abs_subdom_results_list,
    max_rel_subdom_results_list,
    sum_rel_subdom_results_list,
]

# ======================================================================================================================================

for metric_name, metric_results in zip(subdom_metric_names, subdom_metrics_list):

    # Find the configuration with the lowest MSE
    min_metric_idx = np.argmin(metric_results)
    min_metric = metric_results[min_metric_idx]
    best_config = weight_adjustment_strings[min_metric_idx]

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(14, 8))

    # Create color array - highlight the best configuration
    colors = ["lightblue"] * len(weight_adjustment_strings)
    colors[min_metric_idx] = "green"

    # Create bar plot
    x_pos = np.arange(len(weight_adjustment_strings))
    bars = ax.bar(x_pos, metric_results, color=colors, edgecolor="black")

    # Add value labels on top of bars
    for i, (bar, metric) in enumerate(zip(bars, metric_results)):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + 0.001,
            f"{metric:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    # Customize the plot
    ax.set_xlabel("Hyperparameter Configuration", fontsize=12)
    ax.set_ylabel(metric_name, fontsize=12)
    ax.set_title("Hyperparameter Sweep Results", fontsize=14, fontweight="bold")

    # Set x-ticks to show configuration strings
    ax.set_xticks(x_pos)
    ax.set_xticklabels(weight_adjustment_strings, rotation=45, ha="right", fontsize=10)

    # Highlight the best configuration bar
    bars[min_metric_idx].set_edgecolor("darkgreen")
    bars[min_metric_idx].set_linewidth(2)

    # Add a horizontal line at the minimum MSE for reference
    ax.axhline(
        y=min_metric,
        color="red",
        linestyle="--",
        alpha=0.5,
        label=f"Minimum {metric_name}: {min_metric:.4f}",
    )

    # Add legend
    ax.legend()

    # Add text annotation for the best configuration
    ax.text(
        0.02,
        0.98,
        f"Best Config (Index {min_metric_idx}):\n{best_config}\n{metric_name}: {min_metric:.4f}",
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
    )

    # Add grid for better readability
    ax.grid(True, alpha=0.3, axis="y")

    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    plt.show()


KeyError: 'max_abs_subdominance'

In [ ]:
hyperparameter_comp_datasets = [
    "ACSIncome__CA",
    "ACSIncome__IL",
]

# Make graphs comparing the best hyperparameter configurations for the main datasets versus our "validation" datasets
# The goal is to show that our identified best hyperparameter configuration generalizes well to unseen datasets

# List of subdominance metrics
subdom_metric_names = ["max_abs_subdominance", "sum_abs_subdominance", "max_rel_subdominance", "sum_rel_subdominance"]

dataset_metrics_list = []

for dataset in hyperparameter_comp_datasets:
    all_subdom_metrics_list = []
    max_abs_subdom_results_list = []
    sum_abs_subdom_results_list = []
    max_rel_subdom_results_list = []
    sum_rel_subdom_results_list = []

    weight_adjustment_strings = []
    for weight_adjustment in adjusted_weights:
        weight_adjustment_strings.append(str(weight_adjustment))
        max_abs_subdom_list = []
        sum_abs_subdom_list = []
        max_rel_subdom_list = []
        sum_rel_subdom_list = []
        for biased_type in biased_types:
            max_abs_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["max_abs_subdominance"]
                .item() # TODO: is .item() correct, or should it be .to_numpy() 
            )
            sum_abs_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["sum_abs_subdominance"]
                .item()
            )
            max_rel_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["max_rel_subdominance"]
                .item()
            )
            sum_rel_subdom_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc["sum_rel_subdominance"]
                .item()
            )                

        max_abs_subdom_results_list.append(np.average(max_abs_subdom_list))
        sum_abs_subdom_results_list.append(np.average(sum_abs_subdom_list))
        max_rel_subdom_results_list.append(np.average(max_rel_subdom_list))
        sum_rel_subdom_results_list.append(np.average(sum_rel_subdom_list))

    all_subdom_metrics_list.append(max_abs_subdom_results_list)
    all_subdom_metrics_list.append(sum_abs_subdom_results_list)
    all_subdom_metrics_list.append(max_rel_subdom_results_list)
    all_subdom_metrics_list.append(sum_rel_subdom_results_list)
    dataset_metrics_list.append(all_subdom_metrics_list)

# ======================================================================================================================================

for dataset_name, dataset_metrics in zip(hyperparameter_comp_datasets, dataset_metrics_list):
    for metric_name, metric_results in zip(subdom_metric_names, dataset_metrics):

        # Find the configuration with the lowest MSE
        min_metric_idx = np.argmin(metric_results)
        min_metric = metric_results[min_metric_idx]
        best_config = weight_adjustment_strings[min_metric_idx]

        # Create figure and axis
        fig, ax = plt.subplots(figsize=(14, 8))

        # Create color array - highlight the best configuration
        colors = ["lightblue"] * len(weight_adjustment_strings)
        colors[min_metric_idx] = "green"

        # Create bar plot
        x_pos = np.arange(len(weight_adjustment_strings))
        bars = ax.bar(x_pos, metric_results, color=colors, edgecolor="black")

        # Add value labels on top of bars
        for i, (bar, metric) in enumerate(zip(bars, metric_results)):
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                height + 0.001,
                f"{metric:.4f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

        # Customize the plot
        ax.set_xlabel("Hyperparameter Configuration", fontsize=12)
        ax.set_ylabel(metric_name, fontsize=12)
        ax.set_title("Hyperparameter Sweep Results", fontsize=14, fontweight="bold")

        # Set x-ticks to show configuration strings
        ax.set_xticks(x_pos)
        ax.set_xticklabels(weight_adjustment_strings, rotation=45, ha="right", fontsize=10)

        # Highlight the best configuration bar
        bars[min_metric_idx].set_edgecolor("darkgreen")
        bars[min_metric_idx].set_linewidth(2)

        # Add a horizontal line at the minimum MSE for reference
        ax.axhline(
            y=min_metric,
            color="red",
            linestyle="--",
            alpha=0.5,
            label=f"Minimum {metric_name}: {min_metric:.4f}",
        )

        # Add legend
        ax.legend()

        # Add text annotation for the best configuration
        ax.text(
            0.02,
            0.98,
            f"Best Config (Index {min_metric_idx}):\n{best_config}\n{metric_name}: {min_metric:.4f}",
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
        )

        # Add grid for better readability
        ax.grid(True, alpha=0.3, axis="y")

        # Adjust layout to prevent label cutoff
        plt.tight_layout()
        plt.show()


In [ ]:
selected_expert = "OptAcc"
performance_metric_weight = 0.5
fairness_metric_weight = 1 - performance_metric_weight

# 0.0 -	    0.7 0.8 0.3 0.8
# 0.25 -	0.7 0.8 0.3 0.8
# 0.5 -	    0.2 0.8 0.3 0.8
# 0.75 -	0.0 0.8 0.3 0.8
# 1.0 -	    0.0 0.0 0.9 0.4-0.9


unbiased_types = [
    (),
]
biased_types = [
    ("threshold_swapping",),
    ("unbalanced_redlining",),
    ("balanced_redlining",),
    ("perfectly_balanced_redlining",),
]

unadjusted_weights = ()
# adjusted_weights = ()
# adjusted_weights = (("mul_negative_weights", 0.5),)
# adjusted_weights = ("sqrt_negative_weights",)
# adjusted_weights = ("ln_negative_weights",)
adjusted_weights = (
    (("mul_negative_weights", 0.0),),
    (("mul_negative_weights", 0.1),),
    (("mul_negative_weights", 0.2),),
    (("mul_negative_weights", 0.3),),
    (("mul_negative_weights", 0.4),),
    (("mul_negative_weights", 0.5),),
    (("mul_negative_weights", 0.6),),
    (("mul_negative_weights", 0.7),),
    (("mul_negative_weights", 0.8),),
    (("mul_negative_weights", 0.9),),
)

# The goal is to graph the average fairness expectation error of each weight adjustment on our hyperparameter sweep
# identify what hyperparameter is best

# average across all bias types
# Average across all datasets

# Weighted average with acc metric
perf_metrics = ["Acc"]
# Average across all fairness metrics ("DemPar", "EqOpp", "TNRPar")
fair_metrics = ["DemPar", "EqOpp", "TNRPar"]
all_metrics = perf_metrics + fair_metrics
metric_weights = []
for weight in perf_metrics:
    metric_weights.append(performance_metric_weight)
for weight in fair_metrics:
    metric_weights.append(fairness_metric_weight)

unbiased_metric_names = []
biased_metric_names = []
unbiased_perf_metric_names = []
biased_perf_metric_names = []
unbiased_fair_metric_names = []
biased_fair_metric_names = []
for metric in all_metrics:
    unbiased_metric_name = "muE_unbiased_" + metric + "_mean"
    biased_metric_name = "muL_best_" + metric
    unbiased_metric_names.append(unbiased_metric_name)
    biased_metric_names.append(biased_metric_name)
# for metric in perf_metrics:
#     unbiased_metric_name = "muE_unbiased_" + metric + "_mean"
#     biased_metric_name = "muL_best_" + metric
#     unbiased_perf_metric_names.append(unbiased_metric_name)
#     biased_perf_metric_names.append(biased_metric_name)
# for metric in fair_metrics:
#     unbiased_metric_name = "muE_unbiased_" + metric + "_mean"
#     biased_metric_name = "muL_best_" + metric
#     unbiased_fair_metric_names.append(unbiased_metric_name)
#     biased_fair_metric_names.append(biased_metric_name)

# abs(unbiased_data - debiased_data)
mae_results_list = []
# (unbiased_data - debiased_data) ** 2
mse_results_list = []
# unbiased_data - debiased_data
diff_results_list = []
# wherever (unbiased_data - debiased_data) > 0:
# unbiased_data - debiased_data
# otherwise:
# 0
below_results_list = []


weight_adjustment_strings = []
for weight_adjustment in adjusted_weights:
    weight_adjustment_strings.append(str(weight_adjustment))
    unbiased_type_results_list = []
    biased_type_results_list = []
    for dataset in averaged_info:
        for unbiased_type in unbiased_types:
            unbiased_type_results_list.append(
                averaged_data[dataset][selected_expert][unbiased_type][
                    unadjusted_weights
                ]
                .loc[unbiased_metric_names]
                .to_numpy()
            )
        for biased_type in biased_types:
            biased_type_results_list.append(
                averaged_data[dataset][selected_expert][biased_type][
                    weight_adjustment
                ]
                .loc[biased_metric_names]
                .to_numpy()
            )

    mae_list = []
    mse_list = []
    diff_list = []
    below_list = []

    for unbiased_data, debiased_data in zip(unbiased_type_results_list, biased_type_results_list):
        # These values are in terms of loss: closer to 0 is better
        abs_error = np.abs(unbiased_data - debiased_data)
        weighted_diff = abs_error * metric_weights
        weighted_mae = np.sum(weighted_diff) / np.sum(metric_weights)
        mae_list.append(weighted_mae)

        sq_error = (unbiased_data - debiased_data) ** 2
        weighted_diff = sq_error * metric_weights
        weighted_mse = np.sum(weighted_diff) / np.sum(metric_weights)
        mse_list.append(weighted_mse)

        val_diff = unbiased_data - debiased_data
        weighted_diff = val_diff * metric_weights
        weighted_diff = np.sum(weighted_diff) / np.sum(metric_weights)
        diff_list.append(weighted_diff)

        val_below = unbiased_data - debiased_data # This is the amount the unbiased classifier is surpassing us
        val_below[val_below <= 0] = 0 # if we are surpassing the unbiased classifier, don't decrease loss for that (only penalize failing to reach parity, don't reward surpassing parity)
        weighted_below = val_below * metric_weights
        weighted_below = np.sum(weighted_below) / np.sum(metric_weights)
        below_list.append(weighted_below)

    mae_results_list.append(np.average(mae_list))
    mse_results_list.append(np.average(mse_list))
    diff_results_list.append(np.average(diff_list))
    below_results_list.append(np.average(below_list))

# ======================================================================================================================================

# Find the configuration with the lowest MSE
min_metric_idx = np.argmin(mse_results_list)
min_metric = mse_results_list[min_metric_idx]
best_config = weight_adjustment_strings[min_metric_idx]

# Create figure and axis
fig, ax = plt.subplots(figsize=(14, 8))

# Create color array - highlight the best configuration
colors = ["lightblue"] * len(weight_adjustment_strings)
colors[min_metric_idx] = "green"

# Create bar plot
x_pos = np.arange(len(weight_adjustment_strings))
bars = ax.bar(x_pos, mse_results_list, color=colors, edgecolor="black")

# Add value labels on top of bars
for i, (bar, mse) in enumerate(zip(bars, mse_results_list)):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + 0.001,
        f"{mse:.4f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Customize the plot
ax.set_xlabel("Hyperparameter Configuration", fontsize=12)
ax.set_ylabel("Mean Squared Error", fontsize=12)
ax.set_title("Hyperparameter Sweep Results", fontsize=14, fontweight="bold")

# Set x-ticks to show configuration strings
ax.set_xticks(x_pos)
ax.set_xticklabels(weight_adjustment_strings, rotation=45, ha="right", fontsize=10)

# Highlight the best configuration bar
bars[min_metric_idx].set_edgecolor("darkgreen")
bars[min_metric_idx].set_linewidth(2)

# Add a horizontal line at the minimum MSE for reference
ax.axhline(
    y=min_metric,
    color="red",
    linestyle="--",
    alpha=0.5,
    label=f"Minimum MSE: {min_metric:.4f}",
)

# Add legend
ax.legend()

# Add text annotation for the best configuration
ax.text(
    0.02,
    0.98,
    f"Best Config (Index {min_metric_idx}):\n{best_config}\nMSE: {min_metric:.4f}",
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
)

# Add grid for better readability
ax.grid(True, alpha=0.3, axis="y")

# Adjust layout to prevent label cutoff
plt.tight_layout()
plt.show()

# ======================================================================================================================================

# Find the configuration with the lowest mae
min_mae_idx = np.argmin(mae_results_list)
min_mae = mae_results_list[min_mae_idx]
best_config = weight_adjustment_strings[min_mae_idx]

# Create figure and axis
fig, ax = plt.subplots(figsize=(14, 8))

# Create color array - highlight the best configuration
colors = ["lightblue"] * len(weight_adjustment_strings)
colors[min_mae_idx] = "green"

# Create bar plot
x_pos = np.arange(len(weight_adjustment_strings))
bars = ax.bar(x_pos, mae_results_list, color=colors, edgecolor="black")

# Add value labels on top of bars
for i, (bar, mae) in enumerate(zip(bars, mae_results_list)):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + 0.001,
        f"{mae:.4f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Customize the plot
ax.set_xlabel("Hyperparameter Configuration", fontsize=12)
ax.set_ylabel("Mean Absolute Error", fontsize=12)
ax.set_title("Hyperparameter Sweep Results", fontsize=14, fontweight="bold")

# Set x-ticks to show configuration strings
ax.set_xticks(x_pos)
ax.set_xticklabels(weight_adjustment_strings, rotation=45, ha="right", fontsize=10)

# Highlight the best configuration bar
bars[min_mae_idx].set_edgecolor("darkgreen")
bars[min_mae_idx].set_linewidth(2)

# Add a horizontal line at the minimum mae for reference
ax.axhline(
    y=min_mae,
    color="red",
    linestyle="--",
    alpha=0.5,
    label=f"Minimum mae: {min_mae:.4f}",
)

# Add legend
ax.legend()

# Add text annotation for the best configuration
ax.text(
    0.02,
    0.98,
    f"Best Config (Index {min_mae_idx}):\n{best_config}\nmae: {min_mae:.4f}",
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
)

# Add grid for better readability
ax.grid(True, alpha=0.3, axis="y")

# Adjust layout to prevent label cutoff
plt.tight_layout()
plt.show()

# ======================================================================================================================================

# Find the configuration with the lowest diff
min_diff_idx = np.argmin(diff_results_list)
min_diff = diff_results_list[min_diff_idx]
best_config = weight_adjustment_strings[min_diff_idx]

# Create figure and axis
fig, ax = plt.subplots(figsize=(14, 8))

# Create color array - highlight the best configuration
colors = ["lightblue"] * len(weight_adjustment_strings)
colors[min_diff_idx] = "green"

# Create bar plot
x_pos = np.arange(len(weight_adjustment_strings))
bars = ax.bar(x_pos, diff_results_list, color=colors, edgecolor="black")

# Add value labels on top of bars
for i, (bar, diff) in enumerate(zip(bars, diff_results_list)):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + 0.001,
        f"{diff:.4f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Customize the plot
ax.set_xlabel("Hyperparameter Configuration", fontsize=12)
ax.set_ylabel("Relative Metric Difference", fontsize=12)
ax.set_title("Hyperparameter Sweep Results", fontsize=14, fontweight="bold")

# Set x-ticks to show configuration strings
ax.set_xticks(x_pos)
ax.set_xticklabels(weight_adjustment_strings, rotation=45, ha="right", fontsize=10)

# Highlight the best configuration bar
bars[min_diff_idx].set_edgecolor("darkgreen")
bars[min_diff_idx].set_linewidth(2)

# Add a horizontal line at the minimum diff for reference
ax.axhline(
    y=min_diff,
    color="red",
    linestyle="--",
    alpha=0.5,
    label=f"Minimum diff: {min_diff:.4f}",
)

# Add legend
ax.legend()

# Add text annotation for the best configuration
ax.text(
    0.02,
    0.98,
    f"Best Config (Index {min_diff_idx}):\n{best_config}\ndiff: {min_diff:.4f}",
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
)

# Add grid for better readability
ax.grid(True, alpha=0.3, axis="y")

# Adjust layout to prevent label cutoff
plt.tight_layout()
plt.show()

# ======================================================================================================================================

# Find the configuration with the lowest below
min_below_idx = np.argmin(below_results_list)
min_below = below_results_list[min_below_idx]
best_config = weight_adjustment_strings[min_below_idx]

# Create figure and axis
fig, ax = plt.subplots(figsize=(14, 8))

# Create color array - highlight the best configuration
colors = ["lightblue"] * len(weight_adjustment_strings)
colors[min_below_idx] = "green"

# Create bar plot
x_pos = np.arange(len(weight_adjustment_strings))
bars = ax.bar(x_pos, below_results_list, color=colors, edgecolor="black")

# Add value labels on top of bars
for i, (bar, below) in enumerate(zip(bars, below_results_list)):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + 0.001,
        f"{below:.4f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Customize the plot
ax.set_xlabel("Hyperparameter Configuration", fontsize=12)
ax.set_ylabel("Relative Metric (Only When Below)", fontsize=12)
ax.set_title("Hyperparameter Sweep Results", fontsize=14, fontweight="bold")

# Set x-ticks to show configuration strings
ax.set_xticks(x_pos)
ax.set_xticklabels(weight_adjustment_strings, rotation=45, ha="right", fontsize=10)

# Highlight the best configuration bar
bars[min_below_idx].set_edgecolor("darkgreen")
bars[min_below_idx].set_linewidth(2)

# Add a horizontal line at the minimum below for reference
ax.axhline(
    y=min_below,
    color="red",
    linestyle="--",
    alpha=0.5,
    label=f"Minimum below: {min_below:.4f}",
)

# Add legend
ax.legend()

# Add text annotation for the best configuration
ax.text(
    0.02,
    0.98,
    f"Best Config (Index {min_below_idx}):\n{best_config}\nbelow: {min_below:.4f}",
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
)

# Add grid for better readability
ax.grid(True, alpha=0.3, axis="y")

# Adjust layout to prevent label cutoff
plt.tight_layout()
plt.show()

In [ ]:
measurable_experts = [
    "OptAcc",
    "HardtDemPar",
    "HardtEqOpp",
    "HardtTNRPar",
    "HardtFPRPar",
    "HardtFNRPar",
]
unbiased_types = ()
# biased_types = ("perfectly_balanced_redlining",)
biased_types = ("balanced_redlining",)
# biased_types = ("unbalanced_redlining",)
# biased_types = ("threshold_swapping",)

unadjusted_weights = ()
adjusted_weights = (("mul_negative_weights", 0.2),)


diff_metric_df_list = []

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
# data_config_list.append(("muL_best_", "", biased_types, unadjusted_weights))  # Original IRL FE
data_config_list.append(
    ("muL_best_", "", biased_types, adjusted_weights)
)  # Zeroed IRL FE
measurable_metrics = [
    "Acc",
    "DemPar",
    "EqOpp",
    "TNRPar",
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
# measurable_metrics = ["Acc", "DemPar", "EqOpp", "TNRPar", "FPRPar", "FNRPar", "PR_Z0", "PR_Z1", "NR_Z0", "NR_Z1", "TPR_Z0", "TPR_Z1", "TNR_Z0", "TNR_Z1", "FPR_Z0", "FPR_Z1", "FNR_Z0", "FNR_Z1"]
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

selected_expert = "OptAcc"
diff_metric_list = []
dataset_list = []
for dataset in averaged_info:
    dataset_list.append(dataset)
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[0]
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
diff_metric_df_list.append(diff_metric_df)
df1 = diff_metric_df

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
# data_config_list.append(("muE_", "_mean", biased_types, unadjusted_weights)) # Biased Demo FE
data_config_list.append(
    ("muE_unbiased_", "_mean", unbiased_types, unadjusted_weights)
)  # Raw Data FE
measurable_metrics = [
    "Acc",
    "DemPar",
    "EqOpp",
    "TNRPar",
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
# measurable_metrics = ["Acc", "DemPar", "EqOpp", "TNRPar", "FPRPar", "FNRPar", "PR_Z0", "PR_Z1", "NR_Z0", "NR_Z1", "TPR_Z0", "TPR_Z1", "TNR_Z0", "TNR_Z1", "FPR_Z0", "FPR_Z1", "FNR_Z0", "FNR_Z1"]
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

selected_expert = "OptAcc"
diff_metric_list = []
for dataset in averaged_info:
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[0]
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
diff_metric_df_list.append(diff_metric_df)
df2 = diff_metric_df

# -------------------------------------------------------------------
# PLOT CONFIGURATION
# -------------------------------------------------------------------
n_rows = len(df1)  # number of samples per column
n_cols = len(df1.columns)  # number of metrics/columns
colors = {"df1": "tab:blue", "df2": "tab:red"}  # distinct colors
alpha = 0.5  # transparency (0 = fully transparent, 1 = opaque)
bar_width = 0.8  # width of each bar
gap_between_columns = 1.0  # horizontal gap BETWEEN metric groups
label_offset = (
    0.05  # Space between lowest bar bottom and label (as fraction of bar height)
)

fig, ax = plt.subplots(figsize=(14, 7))

# -------------------------------------------------------------------
# PLOT BARS
# -------------------------------------------------------------------
for col_idx, col_name in enumerate(df1.columns):

    # Start position for this metric group on the x‑axis
    start_x = col_idx * (n_rows + gap_between_columns)

    # Extract the values for this column from BOTH DataFrames
    vals_df1 = df1[col_name].values
    vals_df2 = df2[col_name].values

    # Plot every sample (row) in this column
    for row_idx in range(n_rows):
        x_pos = start_x + row_idx

        # Get the row label (index name)
        row_label = df1.index[row_idx]

        # Current values for both bars
        val1 = vals_df1[row_idx]
        val2 = vals_df2[row_idx]

        # -------------------------------------------------------------------
        # CALCULATE SAFE LABEL POSITION (Handles Positive & Negative Data)
        # -------------------------------------------------------------------
        # Find the absolute bottom (most negative value) between the two bars
        lowest_bottom = min(val1, val2, 0.0)

        # Determine offset distance based on bar height to keep proportional spacing
        # Use max absolute height to ensure enough space regardless of bar size
        max_height = max(abs(val1), abs(val2))
        padding_distance = max(0.01 * max_height, 0.01)

        # Calculate final y-position for label (below the lowest bar)
        label_y = lowest_bottom - padding_distance

        # Bar for DataFrame 1
        ax.bar(
            x_pos,
            val1,
            width=bar_width,
            alpha=alpha,
            color=colors["df1"],
            edgecolor="black",
        )

        # Bar for DataFrame 2 (OVERLAPS with DF1 at the SAME x_pos!)
        ax.bar(
            x_pos,
            val2,
            width=bar_width,
            alpha=alpha,
            color=colors["df2"],
            edgecolor="black",
        )

        # Add Row Index Label Below the Bars (Safe for Negative Data)
        ax.text(
            x_pos,
            label_y,
            row_label,
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            rotation=90,
            color="darkgray",
            bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"),
        )

# -------------------------------------------------------------------
# X‑AXIS LABELS (one tick per metric group)
# -------------------------------------------------------------------
# Place a tick at the *center* of each metric group
tick_positions = [
    col_idx * (n_rows + gap_between_columns) + (n_rows - 1) / 2
    for col_idx in range(n_cols)
]
ax.set_xticks(tick_positions)
ax.set_xticklabels(df1.columns, fontsize=11, fontweight="bold", rotation=90)

# -------------------------------------------------------------------
# FINAL TOUCHES
# -------------------------------------------------------------------
ax.set_ylabel("Feature Expectation", fontsize=11)
ax.set_title(
    "Comparison of How Feature Expectations Changed for the OptAcc Expert",
    fontsize=13,
    fontweight="bold",
)
ax.legend(
    ["Feature Expectation for Zeroed Weights", "Feature Expectation Before Bias Added"],
    loc="upper right",
    framealpha=0.5,
    fancybox=True,
)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
data_config_list.append(
    ("muL_best_", "", biased_types, unadjusted_weights)
)  # Original IRL FE
data_config_list.append(
    ("muL_best_", "", biased_types, adjusted_weights)
)  # Zeroed IRL FE
measurable_metrics = [
    "Acc",
    "DemPar",
    "EqOpp",
    "TNRPar",
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
# measurable_metrics = ["Acc", "DemPar", "EqOpp", "TNRPar", "FPRPar", "FNRPar", "PR_Z0", "PR_Z1", "NR_Z0", "NR_Z1", "TPR_Z0", "TPR_Z1", "TNR_Z0", "TNR_Z1", "FPR_Z0", "FPR_Z1", "FNR_Z0", "FNR_Z1"]
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

selected_expert = "OptAcc"
diff_metric_list = []
for dataset in averaged_info:
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][data_config[2]][
            data_config[3]
        ]
        try:
            metric_results = exp_results.loc[measurable_metric]
        except:
            pass
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[1].sub(metric_series_list[0])
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1)
mean_diff_metric_series = diff_metric_df.mean(axis=1)

std_dev = np.std(diff_metric_df, axis=1, ddof=1)

fig, ax = plt.subplots(figsize=(20, 6))

col_names = diff_metric_list[0].index.tolist()
num_arrays = len(diff_metric_list)
array_length = len(diff_metric_list[0].index)

indices = np.arange(num_arrays)

plt.bar(
    mean_diff_metric_series.index,
    mean_diff_metric_series.values,
    yerr=std_dev,
    capsize=5,
    color="skyblue",
    ecolor="black",
)

ax.set_xlabel("Feature Expectation Types")
ax.set_ylabel("Learned Feature Expectation Value Difference (Biased - Original)")
ax.set_title(
    f"Feature Expectation Differences for OptAcc Expert Averaged Across All Datasets"
)

plt.tight_layout()
plt.show()

In [ ]:
diff_metric_df_list = []

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
data_config_list.append(
    ("muL_best_", "", biased_types, unadjusted_weights)
)  # Original IRL FE
data_config_list.append(
    ("muL_best_", "", biased_types, adjusted_weights)
)  # Zeroed IRL FE
measurable_metrics = [
    "Acc",
    "DemPar",
    "EqOpp",
    "TNRPar",
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
# measurable_metrics = ["Acc", "DemPar", "EqOpp", "TNRPar", "FPRPar", "FNRPar", "PR_Z0", "PR_Z1", "NR_Z0", "NR_Z1", "TPR_Z0", "TPR_Z1", "TNR_Z0", "TNR_Z1", "FPR_Z0", "FPR_Z1", "FNR_Z0", "FNR_Z1"]
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

selected_expert = "OptAcc"
diff_metric_list = []
dataset_list = []
for dataset in averaged_info:
    dataset_list.append(dataset)
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[1].sub(metric_series_list[0])
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
diff_metric_df_list.append(diff_metric_df)
df1 = diff_metric_df

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
data_config_list.append(
    ("muE_", "_mean", biased_types, unadjusted_weights)
)  # Biased Demo FE
data_config_list.append(
    ("muE_unbiased_", "_mean", unbiased_types, unadjusted_weights)
)  # Raw Data FE
measurable_metrics = [
    "Acc",
    "DemPar",
    "EqOpp",
    "TNRPar",
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
# measurable_metrics = ["Acc", "DemPar", "EqOpp", "TNRPar", "FPRPar", "FNRPar", "PR_Z0", "PR_Z1", "NR_Z0", "NR_Z1", "TPR_Z0", "TPR_Z1", "TNR_Z0", "TNR_Z1", "FPR_Z0", "FPR_Z1", "FNR_Z0", "FNR_Z1"]
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

selected_expert = "OptAcc"
diff_metric_list = []
for dataset in averaged_info:
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[1].sub(metric_series_list[0])
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
diff_metric_df_list.append(diff_metric_df)
df2 = diff_metric_df

# -------------------------------------------------------------------
# PLOT CONFIGURATION
# -------------------------------------------------------------------
n_rows = len(df1)  # number of samples per column
n_cols = len(df1.columns)  # number of metrics/columns
colors = {"df1": "tab:blue", "df2": "tab:red"}  # distinct colors
alpha = 0.5  # transparency (0 = fully transparent, 1 = opaque)
bar_width = 0.8  # width of each bar
gap_between_columns = 1.0  # horizontal gap BETWEEN metric groups
label_offset = (
    0.01  # Space between lowest bar bottom and label (as fraction of bar height)
)

fig, ax = plt.subplots(figsize=(14, 7))

# -------------------------------------------------------------------
# PLOT BARS
# -------------------------------------------------------------------
for col_idx, col_name in enumerate(df1.columns):

    # Start position for this metric group on the x‑axis
    start_x = col_idx * (n_rows + gap_between_columns)

    # Extract the values for this column from BOTH DataFrames
    vals_df1 = df1[col_name].values
    vals_df2 = df2[col_name].values

    # Plot every sample (row) in this column
    for row_idx in range(n_rows):
        x_pos = start_x + row_idx

        # Get the row label (index name)
        row_label = df1.index[row_idx]

        # Current values for both bars
        val1 = vals_df1[row_idx]
        val2 = vals_df2[row_idx]

        # -------------------------------------------------------------------
        # CALCULATE SAFE LABEL POSITION (Handles Positive & Negative Data)
        # -------------------------------------------------------------------
        # Find the absolute bottom (most negative value) between the two bars
        lowest_bottom = min(val1, val2, 0.0)

        # Determine offset distance based on bar height to keep proportional spacing
        # Use max absolute height to ensure enough space regardless of bar size
        max_height = max(abs(val1), abs(val2))
        padding_distance = max(label_offset * max_height, label_offset)

        # Calculate final y-position for label (below the lowest bar)
        label_y = lowest_bottom - padding_distance

        # Bar for DataFrame 1
        ax.bar(
            x_pos,
            val1,
            width=bar_width,
            alpha=alpha,
            color=colors["df1"],
            edgecolor="black",
        )

        # Bar for DataFrame 2 (OVERLAPS with DF1 at the SAME x_pos!)
        ax.bar(
            x_pos,
            val2,
            width=bar_width,
            alpha=alpha,
            color=colors["df2"],
            edgecolor="black",
        )

        # Add Row Index Label Below the Bars (Safe for Negative Data)
        ax.text(
            x_pos,
            label_y,
            row_label,
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            rotation=90,
            color="darkgray",
            bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"),
        )

# -------------------------------------------------------------------
# X‑AXIS LABELS (one tick per metric group)
# -------------------------------------------------------------------
# Place a tick at the *center* of each metric group
tick_positions = [
    col_idx * (n_rows + gap_between_columns) + (n_rows - 1) / 2
    for col_idx in range(n_cols)
]
ax.set_xticks(tick_positions)
ax.set_xticklabels(df1.columns, fontsize=11, fontweight="bold")

# -------------------------------------------------------------------
# FINAL TOUCHES
# -------------------------------------------------------------------
ax.set_ylabel("Feature Expectation", fontsize=11)
ax.set_title(
    "Comparison of How Feature Expectations Changed for the OptAcc Expert",
    fontsize=13,
    fontweight="bold",
)
ax.legend(
    [
        "Feature Expectation Increase After Bias Removed",
        "Feature Expectation Decrease After Bias Added",
    ],
    loc="upper right",
    framealpha=0.9,
    fancybox=True,
    shadow=True,
)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
# data_config_list.append(("muE_", "_mean", biased_types, unadjusted_weights)) # Biased Demo FE
# data_config_list.append(("muE_unbiased_", "_mean", unbiased_types, unadjusted_weights)) # Raw Data FE
# data_config_list.append(("wL_", "", unbiased_types, unadjusted_weights)) # Raw Data IRL Weights
# data_config_list.append(("wL_", "", biased_types, unadjusted_weights)) # Original (Biased Demo) IRL Weights
# data_config_list.append(("wL_", "", biased_types, adjusted_weights)) # Zeroed IRL Weights
# data_config_list.append(("muL_best_", "", biased_types, unadjusted_weights))  # Original IRL FE
data_config_list.append(
    ("muL_best_", "", biased_types, adjusted_weights)
)  # Zeroed IRL FE

measurable_metrics = [
    "Acc",
    "DemPar",
    "EqOpp",
    "TNRPar",
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
# measurable_metrics = ["Acc", "DemPar", "EqOpp", "TNRPar", "FPRPar", "FNRPar", "PR_Z0", "PR_Z1", "NR_Z0", "NR_Z1", "TPR_Z0", "TPR_Z1", "TNR_Z0", "TNR_Z1", "FPR_Z0", "FPR_Z1", "FNR_Z0", "FNR_Z1"]
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

In [ ]:
for dataset in averaged_info:
    expert_list = []
    diff_metric_list = []
    for expert in averaged_info[dataset]:
        expert_list.append(expert)
        metric_series_list = []
        for data_config, measurable_metric in zip(
            data_config_list, measurable_metric_list
        ):
            exp_results = averaged_data[dataset][expert][data_config[2]][data_config[3]]
            metric_results = exp_results.loc[measurable_metric]
            metric_results.index = measurable_metrics
            metric_series_list.append(metric_results)
        if len(metric_series_list) == 1:
            diff_sub_series = metric_series_list[0]
        elif len(metric_series_list) == 2:
            diff_sub_series = metric_series_list[1].sub(metric_series_list[0])
        elif len(metric_series_list) == 4:
            temp = metric_series_list[1].sub(metric_series_list[0])
            temp2 = metric_series_list[3].sub(metric_series_list[2])
            diff_sub_series = temp2.sub(temp)
        diff_metric_list.append(diff_sub_series)

    trans_diff_wL_list = np.array(diff_metric_list)
    trans_diff_wL_list = trans_diff_wL_list.T
    trans_diff_wL_list = [
        trans_diff_wL_list[i] for i in range(trans_diff_wL_list.shape[0])
    ]

    fig, ax = plt.subplots(figsize=(20, 6))

    col_names = diff_metric_list[0].index.tolist()
    num_arrays = len(diff_metric_list)
    array_length = len(diff_metric_list[0].index)

    indices = np.arange(num_arrays)

    bar_width = 0.7 / array_length

    for mul_val, arr in enumerate(trans_diff_wL_list):
        ax.bar(
            indices + mul_val * bar_width,
            arr,
            width=bar_width,
            label=col_names[mul_val],
        )

    ax.set_xlabel("Expert Classifiers")
    ax.set_ylabel("Learned Feature Expectation Value Difference (Biased - Original)")
    ax.set_title(
        f"Feature Expectation Differences for Each Expert On the {dataset} Dataset"
    )
    ax.set_xticks(indices + bar_width * (array_length - 1) / 2)
    ax.set_xticklabels(expert_list)
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
for dataset in averaged_info:
    expert_list = []
    diff_metric_list = []
    for expert in averaged_info[dataset]:
        expert_list.append(expert)
        metric_series_list = []
        for data_config, measurable_metric in zip(
            data_config_list, measurable_metric_list
        ):
            exp_results = averaged_data[dataset][expert][data_config[2]][data_config[3]]
            metric_results = exp_results.loc[measurable_metric]
            metric_results.index = measurable_metrics
            metric_series_list.append(metric_results)
        if len(metric_series_list) == 1:
            diff_sub_series = metric_series_list[0]
        elif len(metric_series_list) == 2:
            diff_sub_series = metric_series_list[1].sub(metric_series_list[0])
        elif len(metric_series_list) == 4:
            temp = metric_series_list[1].sub(metric_series_list[0])
            temp2 = metric_series_list[3].sub(metric_series_list[2])
            diff_sub_series = temp2.sub(temp)
        diff_metric_list.append(diff_sub_series)

    fig, ax = plt.subplots(figsize=(20, 6))

    col_names = diff_metric_list[0].index.tolist()
    num_arrays = len(diff_metric_list)
    array_length = len(diff_metric_list[0].index)

    indices = np.arange(array_length)

    bar_width = 0.7 / num_arrays

    for mul_val, arr in enumerate(diff_metric_list):
        ax.bar(
            indices + mul_val * bar_width,
            arr.to_numpy(),
            width=bar_width,
            label=expert_list[mul_val],
        )

    ax.set_xlabel("Learned Feature Expectation Types")
    ax.set_ylabel("Learned Feature Expectation Value Difference (Biased - Original)")
    ax.set_title(
        f"Grouped Feature Expectation Differences for All Experts On the {dataset} Dataset"
    )
    ax.set_xticks(indices + bar_width * (num_arrays - 1) / 2)
    ax.set_xticklabels(col_names)
    ax.legend()

    plt.tight_layout()
    plt.show()